In [1]:
import importlib
import inspect
import os
from argparse import Namespace
import numpy as np

%load_ext autoreload
%autoreload 2



def get_all_dataset_names():
    return [
        dataset.split(".")[0]
        for dataset in os.listdir("dataset")
        if not dataset.find("__") > -1 and "py" in dataset
    ]


def get_all_datasets():
    datasets = {}
    for dataset in get_all_dataset_names():
        mod = importlib.import_module("dataset." + dataset)
        dataset_classes_name = [
            x
            for x in mod.__dir__()
            if "type" in str(type(getattr(mod, x)))
            and "DatasetBase" in str(inspect.getmro(getattr(mod, x))[1:])
        ]
        for d in dataset_classes_name:
            c = getattr(mod, d)
            datasets[c.NAME] = c
    return datasets


def get_dataset(dataset_name, accelerator, args: Namespace):
    """
    Returns the dataset object.
    """
    datasets_set = get_all_datasets()
    assert (
        dataset_name in datasets_set
    ), f"Dataset {dataset_name} not found. Available datasets: {datasets_set.keys()}"
    return datasets_set[dataset_name](accelerator, args)


In [18]:
from accelerate import Accelerator
accelerator = Accelerator()

# Configure args for Qwen2 model and dataset
args = Namespace()
args.model = "Qwen/Qwen2.5-0.5B"  # Using smaller Qwen2 model for testing
args.dataset = "MMLU-chem"  # Boolean Questions dataset (2-class classification)
args.add_space = False
args.max_seq_len = 300
args.batch_size = 16
args.is_s2s = False  # Qwen2 is decoder-only, not sequence-to-sequence
args.testing_set = "val"

print(f"Using model: {args.model}")
print(f"Using dataset: {args.dataset}")
# Create the dataset
try:
    dataset = get_all_datasets()['mcdataset'](accelerator, args)
    print(f"✅ Successfully created dataset with {dataset.num_labels} labels")
    
    # Get the dataloaders
    dataset.get_loaders()
    print(f"✅ Successfully created dataloaders")
    print(f"📊 Number of training samples: {dataset.num_samples}")
    print(f"🎯 Target token IDs: {dataset.target_ids}")
    
except Exception as e:
    print(f"❌ Error creating dataset: {e}")
    print("Available datasets:", list(get_all_datasets().keys()))

Using model: Qwen/Qwen2.5-0.5B
Using dataset: MMLU-chem
Loaded MMLU-chem dataset.
✅ Successfully created dataset with 4 labels
✅ Successfully created dataloaders
📊 Number of training samples: 96
🎯 Target token IDs: tensor([[32],
        [33],
        [34],
        [35]])


In [19]:
dataset.target_ids

tensor([[32],
        [33],
        [34],
        [35]])

In [21]:
inp , lab, lab_id = next(iter(dataset.train_dataloader))

In [22]:
lab

tensor([3, 2, 3, 1, 1, 1, 0, 3, 2, 3, 3, 1, 0, 3, 3, 2])

In [23]:
lab_id

tensor([35, 34, 35, 33, 33, 33, 32, 35, 34, 35, 35, 33, 32, 35, 35, 34])

In [ ]:
next(iter(dataset.test_dataloader))


({'input_ids': tensor([[151643, 151643, 151643,  ...,    198,  16141,     25],
         [151643, 151643, 151643,  ...,    198,  16141,     25],
         [151643, 151643, 151643,  ...,    198,  16141,     25],
         [151643, 151643, 151643,  ...,    624,  16141,     25]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1]])},
 tensor([3, 2, 3, 1]),
 tensor([35, 34, 35, 33]))

In [105]:
# Explore the dataset structure
if 'dataset' in locals():
    print("=== Dataset Information ===")
    print(f"Dataset type: {type(dataset)}")
    print(f"Number of labels: {dataset.num_labels}")
    print(f"Tokenizer: {type(dataset.tokenizer)}")
    print(f"Tokenizer vocab size: {dataset.tokenizer.vocab_size}")
    
    # Check special tokens
    print("\n=== Special Tokens ===")
    print(f"BOS token: '{dataset.tokenizer.bos_token}' (ID: {dataset.tokenizer.bos_token_id})")
    print(f"EOS token: '{dataset.tokenizer.eos_token}' (ID: {dataset.tokenizer.eos_token_id})")
    print(f"PAD token: '{dataset.tokenizer.pad_token}' (ID: {dataset.tokenizer.pad_token_id})")
    print(f"UNK token: '{dataset.tokenizer.unk_token}' (ID: {dataset.tokenizer.unk_token_id})")
    
    # Target tokens for classification
    print(f"\n=== Target Tokens for Classification ===")
    print(f"Target IDs shape: {dataset.target_ids.shape}")
    print(f"Target token IDs: {dataset.target_ids.tolist()}")
    
    # Decode target tokens
    for i, target_id in enumerate(dataset.target_ids):
        decoded = dataset.tokenizer.decode(target_id, skip_special_tokens=True)
        print(f"Label {i}: token_id={target_id.item()} -> '{decoded}'")
    
    # Show a sample batch from train dataloader
    print("\n=== Sample Training Batch ===")
    try:
        sample_batch = next(iter(dataset.train_dataloader))
        print(f"Batch type: {type(sample_batch)}")
        print(f"Batch length: {len(sample_batch)}")
        
        # Handle BatchEncoding object
        batch_encoding = sample_batch[0]
        labels = sample_batch[1] if len(sample_batch) > 1 else None
        extra = sample_batch[2] if len(sample_batch) > 2 else None
        
        print(f"\nBatchEncoding keys: {list(batch_encoding.keys())}")
        
        # Get input_ids and attention_mask
        input_ids = batch_encoding['input_ids']
        attention_mask = batch_encoding.get('attention_mask', None)
        
        print(f"Input IDs shape: {input_ids.shape}")
        if attention_mask is not None:
            print(f"Attention mask shape: {attention_mask.shape}")
        if labels is not None:
            print(f"Labels shape: {labels.shape}")
            print(f"Sample labels: {labels[:5].tolist()}")
        if extra is not None:
            print(f"Extra tensor shape: {extra.shape}")
        
        # Show first sample
        print(f"\nFirst sample input IDs (first 20): {input_ids[0][:20].tolist()}")
        if attention_mask is not None:
            print(f"First sample attention mask (first 20): {attention_mask[0][:20].tolist()}")
        
        # Decode first sample
        decoded_text = dataset.tokenizer.decode(input_ids[0], skip_special_tokens=False)
        print(f"\nDecoded sample:\n{decoded_text}...")
        
        # Show the actual length vs padded length
        actual_length = attention_mask[0].sum().item() if attention_mask is not None else len(input_ids[0])
        print(f"\nSequence info:")
        print(f"Total length: {len(input_ids[0])}")
        print(f"Actual length (non-padded): {actual_length}")
        
    except Exception as e:
        print(f"Error exploring batch: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ Dataset not created yet. Run the previous cell first.")

=== Dataset Information ===
Dataset type: <class 'dataset.S2SDataset_Classification.S2SDataset_Classification'>
Number of labels: 2
Tokenizer: <class 'transformers.models.qwen2.tokenization_qwen2_fast.Qwen2TokenizerFast'>
Tokenizer vocab size: 151643

=== Special Tokens ===
BOS token: 'None' (ID: None)
EOS token: '<|endoftext|>' (ID: 151643)
PAD token: '<|endoftext|>' (ID: 151643)
UNK token: 'None' (ID: None)

=== Target Tokens for Classification ===
Target IDs shape: torch.Size([2, 1])
Target token IDs: [[2514], [4049]]
Label 0: token_id=2514 -> 'True'
Label 1: token_id=4049 -> 'False'

=== Sample Training Batch ===
Batch type: <class 'tuple'>
Batch length: 3

BatchEncoding keys: ['input_ids', 'attention_mask']
Input IDs shape: torch.Size([4, 300])
Attention mask shape: torch.Size([4, 300])
Labels shape: torch.Size([4])
Sample labels: [1, 1, 1, 1]
Extra tensor shape: torch.Size([4])

First sample input IDs (first 20): [151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 15

In [103]:
sample_batch, label, target_id = next(iter(dataset.train_dataloader))
print(sample_batch['input_ids'].shape)
print(sample_batch['attention_mask'].shape)
print(label)
print(target_id)

torch.Size([4, 300])
torch.Size([4, 300])
tensor([1, 1, 1, 1])
tensor([4049, 4049, 4049, 4049])


In [72]:
dataset.tokenizer("False")

{'input_ids': [4049], 'attention_mask': [1]}

In [ ]:
# Show actual content without padding
if 'dataset' in locals():
    print("=== Dataset Content Analysis ===")
    
    # Get a batch
    sample_batch = next(iter(dataset.train_dataloader))
    batch_encoding = sample_batch[0]
    labels = sample_batch[1]
    
    input_ids = batch_encoding['input_ids']
    attention_mask = batch_encoding['attention_mask']
    
    # Show actual content (where attention_mask = 1)
    for i in range(min(3, len(input_ids))):  # Show first 3 samples
        print(f"\n=== Sample {i+1} ===")
        print(f"Label: {labels[i].item()}")
        
        # Find the actual content (non-padded)
        mask = attention_mask[i]
        actual_tokens = input_ids[i][mask == 1]
        
        print(f"Actual sequence length: {len(actual_tokens)}")
        print(f"Token IDs: {actual_tokens[:15].tolist()}...")
        
        # Decode the actual content
        actual_text = dataset.tokenizer.decode(actual_tokens, skip_special_tokens=True)
        print(f"Decoded text: {actual_text[:200]}...")
        
        # Check if this is a True/False question by looking for the target tokens
        target_true = dataset.target_ids[0].item()  # 3007 for " True"
        target_false = dataset.target_ids[1].item()  # 3557 for " False"
        
        if target_true in actual_tokens:
            print(f"✅ Contains ' True' token ({target_true})")
        if target_false in actual_tokens:
            print(f"❌ Contains ' False' token ({target_false})")

    print(f"\n=== Dataloader Summary ===")
    print(f"✅ Successfully created Qwen2 dataset for BoolQ")
    print(f"📊 Training samples: {dataset.num_samples}")
    print(f"🏷️ Labels: 2 (True/False)")
    print(f"📝 Batch size: {args.batch_size}")
    print(f"📏 Max sequence length: {args.max_seq_len}")
    print(f"🎯 Target tokens: {dataset.target_ids.squeeze().tolist()} -> [' True', ' False']")

=== Dataset Content Analysis ===

=== Sample 1 ===
Label: 1
Actual sequence length: 188
Token IDs: [4418, 279, 21085, 3685, 323, 4226, 279, 3405, 448, 279, 4244, 364, 1866, 6, 476]...
Decoded text: Read the passage below and answer the question with the words 'true' or 'false'.

Passage: Persian (/ˈpɜːrʒən, -ʃən/), also known by its endonym Farsi (فارسی fārsi (fɒːɾˈsiː) ( listen)), is one of the...

=== Sample 2 ===
Label: 1
Actual sequence length: 201
Token IDs: [4418, 279, 21085, 3685, 323, 4226, 279, 3405, 448, 279, 4244, 364, 1866, 6, 476]...
Decoded text: Read the passage below and answer the question with the words 'true' or 'false'.

Passage: Good Samaritan laws offer legal protection to people who give reasonable assistance to those who are, or who ...

=== Sample 3 ===
Label: 1
Actual sequence length: 100
Token IDs: [4418, 279, 21085, 3685, 323, 4226, 279, 3405, 448, 279, 4244, 364, 1866, 6, 476]...
Decoded text: Read the passage below and answer the question with the words 't

### run with stored dataset

In [1]:
from src.qwen_jax import load_params_and_config, QwenForCausalLM
import jax
import jax.numpy as jnp
import numpy as np

%load_ext autoreload
%autoreload 2

params, config = load_params_and_config("datasets/model_exports/qwen2.5_0.5B_float32")
jax_model = QwenForCausalLM(**config)
params

FrozenDict({
    params: {
        embed_tokens: {
            embedding: Array([[-1.00708008e-02,  4.19921875e-02,  9.88769531e-03, ...,
                     9.94873047e-03,  1.51367188e-02, -7.04956055e-03],
                   [-1.46484375e-02,  6.03199005e-05, -1.84326172e-02, ...,
                    -1.69372559e-03,  3.11279297e-03, -8.42285156e-03],
                   [-3.66210938e-02, -9.82666016e-03,  6.98852539e-03, ...,
                    -7.75146484e-03, -1.70898438e-02, -1.04904175e-04],
                   ...,
                   [ 2.77709961e-03, -9.39941406e-03,  4.24194336e-03, ...,
                    -6.56127930e-03, -1.06201172e-02,  1.72119141e-02],
                   [ 2.77709961e-03, -9.39941406e-03,  4.24194336e-03, ...,
                    -6.56127930e-03, -1.06201172e-02,  1.72119141e-02],
                   [ 2.77709961e-03, -9.39941406e-03,  4.24194336e-03, ...,
                    -6.56127930e-03, -1.06201172e-02,  1.72119141e-02]],      dtype=float32),
    

In [7]:
with np.load("datasets/text_classification/boolq/train_data.npz") as data:
    input_ids = jnp.array(data['input_ids'])
    attention_mask = jnp.array(data['attention_mask'])
    labels = jnp.array(data['labels'])
    target_id = jnp.array(data['target_id'])
    all_target_ids = jnp.array(data['target_ids'])

input_ids.shape, attention_mask.shape, labels.shape, target_id.shape, all_target_ids.shape

((9400, 525), (9400, 525), (9400,), (9400,), (2, 1))

In [56]:
# Test forward pass with loaded Qwen model and dataset batch
# Assumes: jax_model, params, input_ids, attention_mask are already defined

# Forward pass (no gradient, inference mode)
outputs, inter = jax_model.apply(params, input_ids[:4], attention_mask[:4], train=False, mutable=['intermediates'])
print("Forward pass output shape:", outputs.shape)
print("Sample", outputs[0][-1])

Forward pass output shape: (4, 69, 151936)
Sample [ 6.14   -0.7095  2.79   ... -3.08   -3.08   -3.08  ]


In [58]:
arr = inter['intermediates']['layers_0']['self_attn']['attn_weights_masked'][0][0, 0, 250:260, 250:260]
arr = np.array(arr)
print(f"{np.array2string(arr, precision=2)}")

[]


In [59]:
arr = inter['intermediates']['layers_0']['self_attn']['attn_weights_prob'][0][0, 0, 250:260, 250:260]
arr = np.array(arr)
print(f"{np.array2string(arr, precision=2)}")

[]


In [60]:
# Test forward pass with loaded Qwen model and dataset batch
# Assumes: jax_model, params, input_ids, attention_mask are already defined

# Forward pass (no gradient, inference mode)
outputs = jax_model.apply(params, input_ids[:4], attention_mask[:4], train=False)
print("Forward pass output shape:", outputs.shape)
print("Sample", outputs[0, -1])

Forward pass output shape: (4, 69, 151936)
Sample [ 6.14   -0.7095  2.79   ... -3.08   -3.08   -3.08  ]


In [69]:


logit = outputs[:,-1, all_target_ids.flatten()]

logit.shape



(4, 2)

In [70]:
labels

Array([1, 0, 0, ..., 0, 0, 1], dtype=int32)

In [89]:
import optax
nll_optax = optax.losses.softmax_cross_entropy_with_integer_labels(logit, labels[:4])
nll_optax

Array([0.0833, 2.678 , 2.217 , 0.0914], dtype=float16)

In [98]:
# verify cross entropy loss
probs = jax.nn.softmax(logit, axis=-1)
nll_normal = -jnp.log(jnp.take_along_axis(probs, labels[:4].reshape(-1, 1), axis=1)).flatten()
print(f"allclose(nll_normal, nll_optax): {jnp.allclose(nll_normal, nll_optax, rtol=1e-2)}")
nll_normal

allclose(nll_normal, nll_optax): True


Array([0.0835 , 2.678  , 2.217  , 0.09143], dtype=float16)

In [14]:
selected = jnp.take_along_axis(outputs[:, -1], target_id[:4].reshape(-1, 1), axis=-1)
selected

Array([[9.555],
       [7.13 ],
       [7.43 ],
       [9.664]], dtype=float16)

In [10]:
all_ids = jnp.unique(target_id)
print("All unique target IDs:", all_ids)
probs = jax.nn.softmax(outputs[:, -1, all_ids], axis=-1)
print(f"Probabilities {probs} shape:{probs.shape}")
# target mask 
target_mask = jnp.argmax(target_id[:4].reshape(-1, 1) == all_ids.reshape(1, -1), axis=-1, keepdims=True)
print("Target mask indices:", target_mask.shape)

jnp.take_along_axis(probs, target_mask, axis=-1)

All unique target IDs: [32 33]
Probabilities [[0.07916 0.921  ]
 [0.06934 0.931  ]
 [0.1082  0.892  ]
 [0.0857  0.914  ]] shape:(4, 2)
Target mask indices: (4, 1)


Array([[0.921  ],
       [0.06934],
       [0.1082 ],
       [0.914  ]], dtype=float16)

In [11]:
selected = jnp.take_along_axis(outputs[:, -1], all_ids.reshape(1, -1), axis=-1)
selected

Array([[7.1  , 9.555],
       [7.13 , 9.73 ],
       [7.43 , 9.54 ],
       [7.297, 9.664]], dtype=float16)

### run with torch

In [12]:
# from transformers import  Qwen2ForCausalLM

# modell = Qwen2ForCausalLM(model.config)

In [ ]:
from transformers import Qwen2ForCausalLM

In [21]:
# Run equivalent forward pass using PyTorch (Transforms Qwen model)
import gc
gc.collect()
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

%load_ext autoreload
%autoreload 2

# Model name used in the torch notebook
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Target Device:", device)

# Load tokenizer and model (trust_remote_code matches torch notebook)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_cfg = {
    'device_map': 'auto',
    'torch_dtype': torch.float16 if torch.cuda.is_available() else torch.float32,
    'trust_remote_code': False,
}
print("Loading PyTorch model (this may print a lot)...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_cfg)
model.eval()

# Prepare inputs (use data loaded earlier in the notebook)
# input_ids, attention_mask, labels, target_id were loaded from the dataset .npz
torch_input_ids = torch.from_numpy(np.array(input_ids[:4])).to(device)
torch_attention_mask = torch.from_numpy(np.array(attention_mask[:4])).to(device)

# Forward pass (no grad)
with torch.no_grad():
    torch_out = model(input_ids=torch_input_ids, attention_mask=torch_attention_mask)
    # torch_out = model(input_ids=torch_input_ids, attention_mask=torch.ones(4,300))

# Get logits (batch, seq_len, vocab)
torch_logits = torch_out.logits if hasattr(torch_out, 'logits') else torch_out[0]
print("Torch logits shape:", torch_logits.shape)

# Select last token logits (equivalent to outputs[:, -1] in JAX run)
torch_last = torch_logits[:, -1, :].cpu().numpy()
print("Selected last-token logits shape (torch):", torch_last.shape)

# If JAX outputs exist, compare numerically (outputs from JAX should be in variable `outputs`)
try:
    jax_last = np.array(outputs[:4, -1, :])
    abs_diff = np.abs(torch_last - jax_last)
    print("Comparison with JAX last-token logits:")
    print("  max abs diff:", float(abs_diff.max()))
    print("  mean abs diff:", float(abs_diff.mean()))
except NameError:
    print("JAX outputs not found in workspace; skipping numeric comparison.")

# Compute probabilities for the small set of target ids used earlier
all_ids = np.unique(np.array(target_id))
print("All unique target IDs:", all_ids)
torch_all_ids = torch.tensor(all_ids, dtype=torch.long).to(device)
torch_probs = torch.softmax(torch_logits[:, -1, :][:, torch_all_ids], dim=-1).cpu().numpy()
print("Probabilities shape (batch, n_targets):", torch_probs.shape)

# Map target_id entries to indices into all_ids (same mapping as JAX code) and pick the prob
torch_target_ids = np.array(target_id[:4]).reshape(-1, 1)
# mapping via argmax on equality
target_mask = (torch_target_ids == all_ids.reshape(1, -1)).argmax(axis=1).reshape(-1, 1)
selected_probs = np.take_along_axis(torch_probs, target_mask, axis=-1).squeeze(-1)
print("Selected target probabilities (per-sample):", selected_probs.tolist())

# Also show logits at the target positions
selected_logits = np.take_along_axis(torch_last, torch_target_ids, axis=-1).squeeze(-1)
print("Selected target logits (per-sample):", selected_logits.tolist())

Target Device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading PyTorch model (this may print a lot)...
Torch logits shape: torch.Size([4, 201, 151936])
Selected last-token logits shape (torch): (4, 151936)
JAX outputs not found in workspace; skipping numeric comparison.
All unique target IDs: [2514 4049]
Probabilities shape (batch, n_targets): (4, 2)
Selected target probabilities (per-sample): [0.826171875, 0.826171875, 0.826171875, 0.826171875]
Selected target logits (per-sample): [2.15625, 2.15625, 2.15625, 2.15625]


#### manual attention

In [23]:
from torch.nn.functional import scaled_dot_product_attention as spda
import numpy as np
query = torch.randn(2,4,8)
key = torch.randn(2,4,8)
value = torch.arange(2*4*2, dtype=torch.float32).reshape(2,4,2)

out = spda(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=True)
print(out.shape)
out

torch.Size([2, 4, 2])


tensor([[[ 0.0000,  1.0000],
         [ 1.2442,  2.2442],
         [ 1.7602,  2.7602],
         [ 3.3093,  4.3093]],

        [[ 8.0000,  9.0000],
         [ 9.2654, 10.2654],
         [ 8.6691,  9.6691],
         [ 9.3762, 10.3762]]])

In [24]:
value

tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.],
         [ 6.,  7.]],

        [[ 8.,  9.],
         [10., 11.],
         [12., 13.],
         [14., 15.]]])

In [25]:
weight = (query @ key.transpose(-2,-1) / np.sqrt(8))
weight += torch.tril(torch.full((4,4), -torch.inf), diagonal=-1).T
weight = torch.nn.Softmax(dim=-1)(weight)
print(weight)
out_manual = weight @ value
print(out_manual)
print("spda is all the same:", torch.allclose(out, out_manual))

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.3779, 0.6221, 0.0000, 0.0000],
         [0.3275, 0.4649, 0.2076, 0.0000],
         [0.0580, 0.4170, 0.3373, 0.1877]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.3673, 0.6327, 0.0000, 0.0000],
         [0.7936, 0.0782, 0.1282, 0.0000],
         [0.5415, 0.3068, 0.0736, 0.0780]]])
tensor([[[ 0.0000,  1.0000],
         [ 1.2442,  2.2442],
         [ 1.7602,  2.7602],
         [ 3.3093,  4.3093]],

        [[ 8.0000,  9.0000],
         [ 9.2654, 10.2654],
         [ 8.6691,  9.6691],
         [ 9.3762, 10.3762]]])
spda is all the same: True


In [32]:
mask = torch.tril(torch.full((4,4), True), diagonal=0)
mask[:,0] = False
print(mask)
print(value)
out = spda(query, key, value, attn_mask=mask, dropout_p=0.0, is_causal=False)
print(f"out: {out} \n shape: {out.shape}")

tensor([[False, False, False, False],
        [False,  True, False, False],
        [False,  True,  True, False],
        [False,  True,  True,  True]])
tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.],
         [ 6.,  7.]],

        [[ 8.,  9.],
         [10., 11.],
         [12., 13.],
         [14., 15.]]])
out: tensor([[[ 0.0000,  0.0000],
         [ 2.0000,  3.0000],
         [ 2.6174,  3.6174],
         [ 3.5131,  4.5131]],

        [[ 0.0000,  0.0000],
         [10.0000, 11.0000],
         [11.2425, 12.2425],
         [11.0019, 12.0019]]]) 
 shape: torch.Size([2, 4, 2])


In [50]:
from transformers.masking_utils import sdpa_mask_recent_torch
import torch

att_mask = torch.tensor([[1,1,1,1],
                           [0,1,1,0]], dtype=torch.bool)

att_mask = sdpa_mask_recent_torch(2, cache_position=torch.arange(4), kv_length=4, attention_mask=att_mask)
att_mask

tensor([[[[ True, False, False, False],
          [ True,  True, False, False],
          [ True,  True,  True, False],
          [ True,  True,  True,  True]]],


        [[[False, False, False, False],
          [False,  True, False, False],
          [False,  True,  True, False],
          [False,  True,  True, False]]]])

In [51]:
att_mask.shape

torch.Size([2, 1, 4, 4])

In [ ]:
attn_bias = torch.zeros(4, 4, dtype=query.dtype, device=query.device)
attn_bias.masked_fill_(att_mask[1,0].logical_not(), float("-inf"))
attn_bias

tensor([[-inf, -inf, -inf, -inf],
        [-inf, 0., -inf, -inf],
        [-inf, 0., 0., -inf],
        [-inf, 0., 0., -inf]])

In [53]:
torch.softmax(attn_bias, dim=-1)

tensor([[   nan,    nan,    nan,    nan],
        [0.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.5000, 0.5000, 0.0000],
        [0.0000, 0.5000, 0.5000, 0.0000]])

In [56]:
att_mask[1,0]

tensor([[False, False, False, False],
        [False,  True, False, False],
        [False,  True,  True, False],
        [False,  True,  True, False]])

In [62]:
import jax.numpy as jnp
caus_mask = jnp.triu(jnp.ones((4, 4), dtype=jnp.float16) * -jnp.inf, k=1)[None, None, :, :]
att_mask = jnp.array(([False, True, True, True],), dtype=bool)[:, None, None, :]
caus_mask = jnp.where(att_mask, caus_mask, -jnp.inf)
caus_mask

Array([[[[-inf, -inf, -inf, -inf],
         [-inf,   0., -inf, -inf],
         [-inf,   0.,   0., -inf],
         [-inf,   0.,   0.,   0.]]]], dtype=float16)

#### continue

In [65]:
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))
print(f"\nDecoded target: {target_id[0]} => {tokenizer.decode(target_id[0], skip_special_tokens=True)}")

Return the label of the correct answer for the question below.

Question: Ian volunteered to eat Dennis's menudo after already having a bowl because _ despised eating intestine.
Choices:
A) Ian
B) Dennis
Answer:

Decoded target: 33 => B


In [66]:
type(model.model.layers[0].self_attn)

transformers.models.qwen2.modeling_qwen2.Qwen2Attention

In [179]:
from torch import nn 

# class ForwardWrapper(nn.Module):
#     def __init__(self, model):
#         super().__init__()
#         self.model = model

#     def forward_wrapper(self, *args, **kwargs):
#         save_['in_args'] = args
#         save_['in_kwargs'] = kwargs
#         return self.model(*args, **kwargs)

save_ = {}
def my_hook(module, in1, in2, in3):
    print(module)
    save_['in2'] = in2
    save_['in1'] = in1
    save_['in3'] = in3

# model.model.layers[0].self_attn = ForwardWrapper(model.model.layers[0].self_attn)

hook_reg = model.model.layers[0].self_attn.register_forward_hook(my_hook, with_kwargs=True)
try:
    with torch.no_grad():
        torch_out = model(input_ids=torch_input_ids, attention_mask=torch_attention_mask)
finally:
    hook_reg.remove()

Qwen2Attention(
  (q_proj): Linear(in_features=896, out_features=896, bias=True)
  (k_proj): Linear(in_features=896, out_features=128, bias=True)
  (v_proj): Linear(in_features=896, out_features=128, bias=True)
  (o_proj): Linear(in_features=896, out_features=896, bias=False)
)


In [194]:
save_['in3'][0][0, 253:260]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0093,  0.0082, -0.0002,  ..., -0.0021, -0.0048,  0.0034],
        [ 0.0094,  0.0002,  0.0060,  ..., -0.0061, -0.0086,  0.0107],
        ...,
        [ 0.0121, -0.0026,  0.0126,  ..., -0.0173, -0.0059,  0.0077],
        [-0.0057,  0.0026,  0.0017,  ..., -0.0049, -0.0058,  0.0156],
        [-0.0120,  0.0034, -0.0035,  ..., -0.0046, -0.0085,  0.0035]],
       device='cuda:0', dtype=torch.float16)

In [67]:
input_ids.shape, attention_mask.shape, labels.shape, target_id.shape

((2556, 300), (2556, 300), (2556,), (2556,))

In [45]:
save_['in3'][0].shape

torch.Size([4, 220, 896])

In [25]:
save_['in2'].keys()

dict_keys(['hidden_states', 'attention_mask', 'position_ids', 'past_key_values', 'use_cache', 'cache_position', 'position_embeddings'])

In [36]:
save_['in2']['position_embeddings'][0].shape

torch.Size([1, 220, 64])

In [49]:
attention_mask[:4].shape

(4, 300)

In [43]:
save_['in2']['hidden_states'].shape

torch.Size([4, 220, 896])

In [ ]:
save_['in2']['attention_mask'][0]

tensor([[[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ...,  True, False, False],
         [False, False, False,  ...,  True,  True, False],
         [False, False, False,  ...,  True,  True,  True]]], device='cuda:0')

#### compare

In [14]:
torch_last
jax_out = jnp.take_along_axis(outputs[:, -1], all_ids.reshape(1, -1), axis=-1)
torch_out = np.take_along_axis(torch_last, all_ids.reshape(1, -1), axis=-1)
print("JAX selected out:", jax_out)
print("Torch selected out:", torch_out)

JAX selected out: [[7.1   9.555]
 [7.13  9.73 ]
 [7.43  9.54 ]
 [7.297 9.664]]
Torch selected out: [[7.094 9.53 ]
 [7.133 9.734]
 [7.45  9.53 ]
 [7.26  9.62 ]]


In [108]:
torch_last
jax_out = jnp.take_along_axis(outputs[:, -1], all_ids.reshape(1, -1), axis=-1)
torch_out = np.take_along_axis(torch_last, all_ids.reshape(1, -1), axis=-1)
print("JAX selected out:", jax_out)
print("Torch selected out:", torch_out)

JAX selected out: [[ 5.887  5.652]
 [10.445  9.234]
 [ 9.36   8.11 ]
 [ 7.477  6.06 ]]
Torch selected out: [[ 8.62   9.96 ]
 [10.164 11.79 ]
 [ 9.03  10.82 ]
 [ 9.15  10.59 ]]


In [106]:
jnp.argmax(attention_mask[:4], axis=-1)

Array([112,  99, 200, 168], dtype=int32)

In [20]:
jnp.broadcast_to(causal_mask, (attention_mask.shape[0], seq_len, seq_len)).shape

(9424, 33, 33)

In [4]:
seq_len = 300
causal_mask = jnp.triu(
                jnp.ones((seq_len, seq_len), dtype=jnp.float16) * -jnp.inf, k=1)
weight = causal_mask + jax.random.normal(key=jax.random.PRNGKey(0), shape=(seq_len, seq_len))
jax.nn.softmax(weight, axis=-1)

Array([[1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [8.8795382e-01, 1.1204623e-01, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.6884800e-01, 1.3232224e-01, 4.9882972e-01, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       ...,
       [4.8701251e-03, 6.8352399e-03, 1.6507380e-02, ..., 2.4016085e-03,
        0.0000000e+00, 0.0000000e+00],
       [1.7637190e-03, 1.3910775e-04, 3.2012344e-03, ..., 2.1198380e-03,
        4.8052880e-04, 0.0000000e+00],
       [1.5128745e-02, 5.9490833e-03, 7.3946971e-03, ..., 2.0531041e-04,
        1.1541211e-03, 3.3987311e-03]], dtype=float32)

In [5]:
causal_mask

Array([[  0., -inf, -inf, ..., -inf, -inf, -inf],
       [  0.,   0., -inf, ..., -inf, -inf, -inf],
       [  0.,   0.,   0., ..., -inf, -inf, -inf],
       ...,
       [  0.,   0.,   0., ...,   0., -inf, -inf],
       [  0.,   0.,   0., ...,   0.,   0., -inf],
       [  0.,   0.,   0., ...,   0.,   0.,   0.]], dtype=float16)

In [6]:
weight = jax.random.normal(key=jax.random.PRNGKey(0), shape=(4, 10, seq_len, seq_len))

In [12]:
attention_mask

Array([[0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1],
       [0, 0, 0, ..., 1, 1, 1]], dtype=int32)

In [15]:

mask = (attention_mask[:4, None, None, :] == 0)
attn_weights2 = jnp.where(mask, -jnp.inf,  weight)
attn_weights2

Array([[[[       -inf,        -inf,        -inf, ..., -0.7116935 ,
          -0.05395174, -2.0926828 ],
         [       -inf,        -inf,        -inf, ..., -0.50935936,
           1.0135212 ,  0.60979146],
         [       -inf,        -inf,        -inf, ...,  1.2272704 ,
          -1.3195466 , -0.32613733],
         ...,
         [       -inf,        -inf,        -inf, ...,  0.12316248,
           0.8117914 , -0.1125572 ],
         [       -inf,        -inf,        -inf, ...,  0.03323483,
          -1.4509728 , -0.11996239],
         [       -inf,        -inf,        -inf, ..., -2.0630805 ,
          -0.33650947,  0.7435536 ]],

        [[       -inf,        -inf,        -inf, ..., -1.3711846 ,
           1.5841771 , -1.0049901 ],
         [       -inf,        -inf,        -inf, ..., -0.00536305,
           0.9028315 ,  1.4512507 ],
         [       -inf,        -inf,        -inf, ...,  1.1094759 ,
          -0.41762683, -0.11658346],
         ...,
         [       -inf,        -inf

In [18]:

attn_weights = jnp.where(attention_mask[:4, None, None, :], weight, -jnp.inf)
attn_weights[0,0]

Array([[       -inf,        -inf,        -inf, ..., -0.7116935 ,
        -0.05395174, -2.0926828 ],
       [       -inf,        -inf,        -inf, ..., -0.50935936,
         1.0135212 ,  0.60979146],
       [       -inf,        -inf,        -inf, ...,  1.2272704 ,
        -1.3195466 , -0.32613733],
       ...,
       [       -inf,        -inf,        -inf, ...,  0.12316248,
         0.8117914 , -0.1125572 ],
       [       -inf,        -inf,        -inf, ...,  0.03323483,
        -1.4509728 , -0.11996239],
       [       -inf,        -inf,        -inf, ..., -2.0630805 ,
        -0.33650947,  0.7435536 ]], dtype=float32)

In [20]:
weight[0,0]

Array([[ 1.6226422 ,  2.0252647 , -0.43359444, ..., -0.7116935 ,
        -0.05395174, -2.0926828 ],
       [ 0.47375467, -1.5962535 , -0.15396847, ..., -0.50935936,
         1.0135212 ,  0.60979146],
       [ 0.399992  , -0.62515265,  0.7018721 , ...,  1.2272704 ,
        -1.3195466 , -0.32613733],
       ...,
       [ 0.8301435 ,  1.1691153 ,  2.0508313 , ...,  0.12316248,
         0.8117914 , -0.1125572 ],
       [-0.15068035, -2.6906116 ,  0.44543153, ...,  0.03323483,
        -1.4509728 , -0.11996239],
       [ 2.236748  ,  1.3033886 ,  1.5209146 , ..., -2.0630805 ,
        -0.33650947,  0.7435536 ]], dtype=float32)

In [17]:
jnp.allclose(attn_weights, attn_weights2)

Array(True, dtype=bool)

In [38]:
causal_mask = causal_mask.at[:, attention_mask[0] == 0].set(-np.inf)
causal_mask

Array([[-inf, -inf, -inf, ..., -inf, -inf, -inf],
       [-inf, -inf, -inf, ..., -inf, -inf, -inf],
       [-inf, -inf, -inf, ..., -inf, -inf, -inf],
       ...,
       [-inf, -inf, -inf, ...,   0., -inf, -inf],
       [-inf, -inf, -inf, ...,   0.,   0., -inf],
       [-inf, -inf, -inf, ...,   0.,   0.,   0.]], dtype=float16)

In [33]:
causal_mask

Array([[  0., -inf, -inf, ..., -inf, -inf, -inf],
       [  0.,   0., -inf, ..., -inf, -inf, -inf],
       [  0.,   0.,   0., ..., -inf, -inf, -inf],
       ...,
       [  0.,   0.,   0., ...,   0., -inf, -inf],
       [  0.,   0.,   0., ...,   0.,   0., -inf],
       [  0.,   0.,   0., ...,   0.,   0.,   0.]], dtype=float16)

In [ ]:
selected = jnp.take_along_axis(outputs[:, -1], all_ids.reshape(1, -1), axis=-1)
selected

Array([[ 5.887,  5.652],
       [10.445,  9.234],
       [ 9.36 ,  8.11 ],
       [ 7.477,  6.06 ]], dtype=float16)

### with subspace model wrapper

In [63]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [64]:
import torch
from transformers import AutoTokenizer

# Model name used in the torch notebook
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Target Device:", device)

# Load tokenizer and model (trust_remote_code matches torch notebook)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Target Device: cuda


In [3]:
import jax
import jax.numpy as jnp
from qwen_fineTuning_PtoC import QwenTextClassificationWrapper
import numpy as np
from flax.core import freeze, unfreeze

%load_ext autoreload
%autoreload 2

# with np.load("datasets/text_classification/boolq/train_data.npz") as data:
with np.load("datasets/text_classification/winogrande_m/train_data.npz") as data:
    input_ids = jnp.array(data['input_ids'])
    attention_mask = jnp.array(data['attention_mask'])
    labels = jnp.array(data['labels'])
    target_id = jnp.array(data['target_id'])
    all_target_ids = jnp.array(data['target_ids'])

input_ids.shape, attention_mask.shape, labels.shape, target_id.shape, all_target_ids.shape

((2500, 69), (2500, 69), (2500,), (2500,), (2, 1))

In [4]:

model = QwenTextClassificationWrapper("datasets/model_exports/qwen2.5_0.5B_bfloat16", target_token_ids=all_target_ids)


params = model.init(None, None)
params = unfreeze(params)
params

{'params': {'embed_tokens': {'embedding': Array([[-0.0100708, 0.0419922, 0.0098877, ..., 0.00994873, 0.0151367,
           -0.00704956],
          [-0.0146484, 6.03199e-05, -0.0184326, ..., -0.00169373,
           0.00311279, -0.00842285],
          [-0.0366211, -0.00982666, 0.00698853, ..., -0.00775146,
           -0.0170898, -0.000104904],
          ...,
          [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
           0.0172119],
          [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
           0.0172119],
          [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
           0.0172119]], dtype=bfloat16)},
  'layers_0': {'input_layernorm': {'weight': Array([-0.0461426, 0.0437012, -0.0244141, 0.0825195, -0.0410156,
           0.0375977, -0.0397949, -0.0267334, -0.105469, -0.0302734,
           0.0441895, -0.0490723, -0.0283203, 0.0354004, 0.0534668, 0.101562,
           0.108398, -0.0476074, 0.223633, 0.108398, 0.112305, -0

In [19]:
import gc
gc.collect()

0

In [17]:
del params
del model

NameError: name 'params' is not defined

In [ ]:
logits = model.apply(params, input_ids[:6], attention_mask[:6], train=False)
logits

Array([[2.14 , 2.742],
       [2.266, 2.498],
       [2.69 , 3.693],
       [2.674, 3.33 ],
       [2.305, 2.213],
       [2.494, 2.258]], dtype=float16)

In [14]:
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))

Return the label of the correct answer for the question below.

Question: Ian volunteered to eat Dennis's menudo after already having a bowl because _ despised eating intestine.
Choices:
A) Ian
B) Dennis
Answer:


In [15]:
tokenizer.decode(all_target_ids.flatten())

'AB'

In [22]:
tokenizer.decode(all_target_ids.flatten()[jnp.argmax(logits, axis=-1)])

'BBBBAA'

In [16]:
tokenizer.decode(jnp.argmax(all[:,-1,:], axis=-1))

' A A B B B B'

In [20]:
tokenizer.decode(target_id[:12])

'BAABBABABABA'

In [17]:
labels[:12]

Array([1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0], dtype=int32)

In [52]:
all_target_ids

Array([[2514],
       [4049]], dtype=int32)

In [31]:
params['params']['layers_0']['self_attn']['q_proj']['kernel'].shape

(2048, 2048)

In [5]:
from src.subspace_curve import CategorySubspace, filter_mask

key = jax.random.PRNGKey(0)
s_model = CategorySubspace(model, k=0)

curve_mask = jax.tree.map(lambda x: False, params)['params']
# curve_mask['layers_0']['self_attn']['q_proj']['kernel'] = True

train_mask = jax.tree.map(lambda x: False, params['params'])
train_mask = filter_mask(train_mask, "self_attn", "q_proj", "kernel", as_curve=True)

s_params = s_model.init_params_from_point(key, params, curve_mask, jitter=0.)

s_model.set_train_mask(train_mask)
s_model(s_params['params'], {}, 0.5, (input_ids[:12], attention_mask[:12]), train=False)


(Array([[7.0625, 9.5625],
        [7.09375, 9.75],
        [7.25, 9.375],
        [7.25, 9.5625],
        [7.875, 10.25],
        [7.875, 10.25],
        [6.8125, 9.3125],
        [6.8125, 8.375],
        [5.34375, 9.1875],
        [5.5625, 9.3125],
        [8.125, 9.6875],
        [8.1875, 9.75]], dtype=bfloat16),
 {})

In [39]:
from qwen_fineTuning_PtoC import setup_metrics

metrics = setup_metrics(s_model, 4, input_ids[:12], attention_mask[:12], labels[:12], 1, use_linspace=True, average=True)
metrics(key, s_params)

{'loss': Array(1.27344, dtype=bfloat16), 'acc': Array(0.5, dtype=float32)}

In [16]:

train_mask = jax.tree.map(lambda x: True, params['params'])
# train_mask = filter_mask(train_mask, "self_attn", "q_proj", "kernel", as_curve=True)

s_model.set_train_mask(train_mask)

In [70]:
from optax import adamw


optimizer = adamw(1e-5)
trainable_params = s_model.only_trainable_params(s_params['params'])
opt_state = optimizer.init(trainable_params)

In [18]:
sub_key = jax.random.PRNGKey(1)
loss, params, opt_state, grad_ = s_model.train_step(
                    sub_key, params, (input_ids[:4], attention_mask[:4]), labels[:4], opt_state, optimizer)
loss

Array(1.82812, dtype=bfloat16)

In [8]:
loss

Array(1.25781, dtype=bfloat16)

In [20]:
grad_['layers_23'].keys()

dict_keys(['input_layernorm', 'mlp', 'post_attention_layernorm', 'self_attn'])

In [26]:
jnp.isfinite(grad_['norm']['weight']).all()

Array(True, dtype=bool)

In [37]:
for k, v in jax.tree.leaves_with_path(grad_['layers_23']):
    path = '/'.join((ki.key for ki in k))
    # print(f"Check {path}, with shape: {v.shape}")
    all_finite = jnp.isfinite(v).all()
    print(f"{all_finite} \tFinite param: {path}")


False 	Finite param: input_layernorm/weight
True 	Finite param: mlp/down_proj/kernel
True 	Finite param: mlp/gate_proj/kernel
True 	Finite param: mlp/up_proj/kernel
True 	Finite param: post_attention_layernorm/weight
False 	Finite param: self_attn/k_proj/bias
False 	Finite param: self_attn/k_proj/kernel
True 	Finite param: self_attn/o_proj/kernel
False 	Finite param: self_attn/q_proj/bias
False 	Finite param: self_attn/q_proj/kernel
True 	Finite param: self_attn/v_proj/bias
True 	Finite param: self_attn/v_proj/kernel


In [29]:
grad_['layers_23']['post_attention_layernorm']

{'weight': Array([-0.0133057, -0.0167236, 0.00799561, -0.0159912, 0.013916,
        0.0162354, -0.022583, -0.00262451, 0.0201416, -0.0412598,
        -0.000946045, -0.000339508, 0.00102234, 0.00549316, 0.00121307,
        -0.0037384, 0.00842285, -0.0088501, 0.000318527, 0.0022583,
        0.00250244, 0.00294495, 0.0177002, -0.0126953, -0.00300598,
        0.00267029, 0.0163574, 0.00308228, -0.00320435, -0.00180817,
        5.98431e-05, -0.0150757, 0.00221252, -0.0141602, -0.0119629,
        0.0167236, 0.0119019, 0.00191498, 0.0197754, -0.0135498, 0.0712891,
        -0.000976562, -0.00133514, 0.0109863, -0.00187683, 0.0154419,
        -0.0383301, -0.00549316, -0.000210762, 0.00921631, -0.0412598,
        -0.0012207, 0.0761719, -0.00125885, -0.046875, -0.0263672,
        -0.00270081, 0.000352859, -0.00215149, -0.00787354, -0.0283203,
        -0.000117302, 0.0493164, 0.00479126, 0.00714111, -0.0078125,
        0.0356445, 0.00823975, -0.0368652, -0.000146866, 0.00260925,
        0.0218506,

In [9]:
grad_

{'embed_tokens': {'embedding': Array([], shape=(0,), dtype=float32)},
 'layers_0': {'input_layernorm': {'weight': Array([], shape=(0,), dtype=float32)},
  'mlp': {'down_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'gate_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'up_proj': {'kernel': Array([], shape=(0,), dtype=float32)}},
  'post_attention_layernorm': {'weight': Array([], shape=(0,), dtype=float32)},
  'self_attn': {'k_proj': {'bias': Array([], shape=(0,), dtype=float32),
    'kernel': Array([], shape=(0,), dtype=float32)},
   'o_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'q_proj': {'bias': Array([], shape=(0,), dtype=float32),
    'kernel': Array([[nan, nan, nan, ..., nan, nan, nan],
           [nan, nan, nan, ..., nan, nan, nan],
           [nan, nan, nan, ..., nan, nan, nan],
           ...,
           [nan, nan, nan, ..., nan, nan, nan],
           [nan, nan, nan, ..., nan, nan, nan],
           [nan, nan, nan, ..., nan, nan, nan]],

In [68]:
from src.subspace_curve import filter_mask
lora_mask = jax.tree.map(lambda x: False, params)['params']
lora_mask = filter_mask(lora_mask, 'self_attn', 'q_proj', 'kernel', as_curve=np.array([0,1]), BoolschenOperator=all)
lora_mask = filter_mask(lora_mask, 'self_attn', 'v_proj', 'kernel', as_curve=np.array([0,1]), BoolschenOperator=all)
train_mask = jax.tree.map(lambda x: False, params)['params']
train_mask = filter_mask(train_mask, 'self_attn', 'q_proj', 'kernel', as_curve=True, BoolschenOperator=all)
train_mask = filter_mask(train_mask, 'self_attn', 'q_proj', 'bias', as_curve=True, BoolschenOperator=all)
train_mask = filter_mask(train_mask, 'self_attn', 'v_proj', 'kernel', as_curve=True, BoolschenOperator=all)
lora_mask

{'embed_tokens': {'embedding': False},
 'layers_0': {'input_layernorm': {'weight': False},
  'mlp': {'down_proj': {'kernel': False},
   'gate_proj': {'kernel': False},
   'up_proj': {'kernel': False}},
  'post_attention_layernorm': {'weight': False},
  'self_attn': {'k_proj': {'bias': False, 'kernel': False},
   'o_proj': {'kernel': False},
   'q_proj': {'bias': False, 'kernel': array([0, 1])},
   'v_proj': {'bias': False, 'kernel': array([0, 1])}}},
 'layers_1': {'input_layernorm': {'weight': False},
  'mlp': {'down_proj': {'kernel': False},
   'gate_proj': {'kernel': False},
   'up_proj': {'kernel': False}},
  'post_attention_layernorm': {'weight': False},
  'self_attn': {'k_proj': {'bias': False, 'kernel': False},
   'o_proj': {'kernel': False},
   'q_proj': {'bias': False, 'kernel': array([0, 1])},
   'v_proj': {'bias': False, 'kernel': array([0, 1])}}},
 'layers_10': {'input_layernorm': {'weight': False},
  'mlp': {'down_proj': {'kernel': False},
   'gate_proj': {'kernel': False},

In [2]:
from src.qwen_jax import QwenAttention
import optax
import jax
import jax.numpy as jnp

key = jax.random.PRNGKey(0)
mm = QwenAttention(
    hidden_size=10,
    num_attention_heads=2,
    num_key_value_heads=2,
    head_dim=10)

seq_len = 4
pp = mm.init(key, jnp.ones((1,seq_len,10)), jnp.ones((1,seq_len)), train=False)

att_mm = jnp.ones((1,seq_len))
att_mm = att_mm.at[:,:2].set(0)
optimizer = optax.adamw(1e-3)
opt_state = optimizer.init(pp['params'])

def loss_fn(params, x, y):
    out = mm.apply({'params': params}, *x, train=True)
    loss = jnp.mean((out - y)**2)
    return loss

loss, grad = jax.value_and_grad(loss_fn)(pp['params'], x=(jnp.ones((1,seq_len,10)), att_mm), y=jnp.ones((1,seq_len,10)))

In [3]:
for k, v in jax.tree.leaves_with_path(grad):
    path = '/'.join((ki.key for ki in k))
    # print(f"Check {path}, with shape: {v.shape}")
    all_finite = jnp.isfinite(v).all()
    print(f"{all_finite} \tFinite param: {path}")

True 	Finite param: k_proj/bias
True 	Finite param: k_proj/kernel
True 	Finite param: o_proj/kernel
True 	Finite param: q_proj/bias
True 	Finite param: q_proj/kernel
True 	Finite param: v_proj/bias
True 	Finite param: v_proj/kernel


In [4]:
grad['q_proj']['kernel']

Array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00, -2.2046152e-09,  1.3198587e-08,
        -8.4492058e-10,  3.6811798e-10, -2.8448994e-09,  3.2378011e-09,
         2.8230716e-09, -5.3187250e-09,  1.6291324e-10,  1.5759724e-08],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00, -2.2046152e-09,  1.3198587e-08,
        -8.4492058e-10,  3.6811798e-10, -2.8448994e-09,  3.2378011e-09,
         2.8230716e-09, -5.3187250e-09,  1.6291324e-10,  1.5759724e-08],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00, -2.2046152e-09,  1.3198587e-08,
        -8.4492058e-10,  3.6811798e-10, -2.8448994e-09,  3.237

In [111]:
k = jax.random.normal(key=jax.random.PRNGKey(0), shape=(4,3))
q = jax.random.normal(key=jax.random.PRNGKey(1), shape=(4,3))
v = jax.random.normal(key=jax.random.PRNGKey(2), shape=(3,4))
def dot_att(params):
    k = params['k']
    q = params['q']
    v = params['v']
    caus_mask = jnp.triu(jnp.ones((4, 4), dtype=jnp.float16) * -jnp.inf, k=1)
    att_mask = jnp.array(([False, True, True, True]), dtype=bool)[None, :]
    caus_mask = jnp.where(att_mask, caus_mask, -jnp.inf)
    print("caus_mask:", att_mask)

    weight = q @ k.T
    # weight += caus_mask
    print("weight:", weight)
    weight = jax.nn.softmax(weight, axis=-1, where=caus_mask > -jnp.inf)
    print("softmax weight:", weight)
    # weight = jnp.where(jnp.isnan(weight), 0.0, weight)
    return(v @ weight).T.sum()

params = {'k': k, 'q': q, 'v': v}
print(dot_att(params))

# jax.grad(dot_att)(params)

caus_mask: [[False  True  True  True]]
weight: [[-0.02008125  0.15924278  0.02803146  0.597225  ]
 [ 2.2494671   0.09107862  0.80152386  2.6181054 ]
 [ 5.6315546   0.29043037 -0.7578958   0.71817684]
 [-0.7458035   0.08586995 -0.69060636 -1.4033378 ]]
softmax weight: [[0.         0.         0.         0.        ]
 [0.         1.         0.         0.        ]
 [0.         0.74045336 0.25954664 0.        ]
 [0.         0.5932692  0.27291816 0.13381264]]
1.6777906


In [118]:
jnp.logical_not(jnp.triu(jnp.ones((4, 4), dtype=bool), k=1))


Array([[ True, False, False, False],
       [ True,  True, False, False],
       [ True,  True,  True, False],
       [ True,  True,  True,  True]], dtype=bool)

In [124]:
k = jax.random.normal(key=jax.random.PRNGKey(0), shape=(2,2,4,3))
q = jax.random.normal(key=jax.random.PRNGKey(1), shape=(2,2,4,3))
v = jax.random.normal(key=jax.random.PRNGKey(2), shape=(2,2,3,4))
def dot_att(params):
    k = params['k']
    q = params['q']
    v = params['v']
    caus_mask = jnp.logical_not(jnp.triu(jnp.ones((4, 4), dtype=bool), k=1))
    att_mask = jnp.array(([False, True, True, True],[False, True, False, True]), dtype=bool)[:, None, None, :]
    caus_mask = jnp.logical_and(att_mask, caus_mask[None, None, :, :])
    print("caus_mask:", caus_mask)

    weight = q @ k.transpose(0,1,3,2)
    weight += caus_mask
    weight = jax.nn.softmax(weight, axis=-1, where=caus_mask)
    print("softmax weight:", weight)
    # weight = jnp.where(jnp.isnan(weight), 0.0, weight)
    return(v @ weight).sum()

params = {'k': k, 'q': q, 'v': v}
print(dot_att(params))

# jax.grad(dot_att)(params)

caus_mask: [[[[False False False False]
   [False  True False False]
   [False  True  True False]
   [False  True  True  True]]]


 [[[False False False False]
   [False  True False False]
   [False  True False False]
   [False  True False  True]]]]
softmax weight: [[[[0.         0.         0.         0.        ]
   [0.         1.         0.         0.        ]
   [0.         0.74045336 0.25954667 0.        ]
   [0.         0.5932692  0.27291816 0.13381264]]

  [[0.         0.         0.         0.        ]
   [0.         1.         0.         0.        ]
   [0.         0.02821684 0.9717831  0.        ]
   [0.         0.02654445 0.91812605 0.0553295 ]]]


 [[[0.         0.         0.         0.        ]
   [0.         1.         0.         0.        ]
   [0.         1.         0.         0.        ]
   [0.         0.35917556 0.         0.6408244 ]]

  [[0.         0.         0.         0.        ]
   [0.         1.         0.         0.        ]
   [0.         1.         0.         0. 

In [128]:
def dot_att(params):
    k = params['k']
    q = params['q']
    v = params['v']
    caus_mask = jnp.triu(jnp.ones((4, 4), dtype=jnp.float16) * -jnp.inf, k=1)
    att_mask = jnp.array(([False, True, True, True]), dtype=bool)[None, :]
    caus_mask = jnp.where(att_mask, caus_mask, -jnp.inf)
    print("caus_mask:", att_mask)

    weight = q @ k.transpose(0,1,3,2)
    weight += caus_mask
    print("weight:", weight)
    weight = jax.nn.softmax(weight, axis=-1)
    print("softmax weight:", weight)
    weight = jnp.where(jnp.isnan(weight), 0.0, weight)
    return(v @ weight).T.sum()

params = {'k': k, 'q': q, 'v': v}
print(dot_att(params))

caus_mask: [[False  True  True  True]]
weight: [[[[       -inf        -inf        -inf        -inf]
   [       -inf  0.09107862        -inf        -inf]
   [       -inf  0.29043037 -0.7578958         -inf]
   [       -inf  0.08586995 -0.69060636 -1.4033378 ]]

  [[       -inf        -inf        -inf        -inf]
   [       -inf  0.6895977         -inf        -inf]
   [       -inf -0.30639654  3.2328172         -inf]
   [       -inf -2.8594403   0.6840737  -2.1249547 ]]]


 [[[       -inf        -inf        -inf        -inf]
   [       -inf  0.39907694        -inf        -inf]
   [       -inf  0.6600964  -1.8249806         -inf]
   [       -inf -0.20281255 -3.1496904   0.37613165]]

  [[       -inf        -inf        -inf        -inf]
   [       -inf -1.2238208         -inf        -inf]
   [       -inf -1.0578043   0.74380887        -inf]
   [       -inf -1.3797495   0.7887331  -2.2611663 ]]]]
softmax weight: [[[[       nan        nan        nan        nan]
   [0.         1.         0. 

In [102]:
grad['q_proj']['kernel'].shape

(10, 20)

In [100]:
pp

{'params': {'q_proj': {'kernel': Array([[ 5.20676136e-01,  1.15865670e-01,  5.31347692e-01,
           -1.55850455e-01, -1.22442886e-01, -1.13146432e-01,
           -3.19558799e-01,  1.31996438e-01, -2.69251078e-01,
            1.41894281e-01, -3.10501814e-01,  2.79387031e-02,
           -2.21937835e-01, -2.91291058e-01, -7.02484429e-01,
           -3.31509262e-01,  5.60321212e-01,  5.19540548e-01,
           -2.08701611e-01, -5.93628548e-02],
          [-3.17534387e-01,  4.07201409e-01, -1.39272034e-01,
            6.21867657e-01, -5.85186422e-01, -2.43102945e-02,
            2.78294384e-01,  4.71750110e-01, -3.46922696e-01,
           -3.07728089e-02, -5.50254405e-01, -3.96591902e-01,
            1.74992085e-01,  2.58092642e-01, -3.61387223e-01,
            6.52949691e-01, -6.46253705e-01,  2.58663684e-01,
            2.20653564e-01, -1.85032859e-01],
          [-6.00100338e-01,  5.12190640e-01,  8.25878158e-02,
           -2.55487323e-01,  2.06106499e-01, -1.34539306e-01,
          

In [56]:
attention_mask.shape

(2500, 69)

In [59]:
seq_len = 69
att_w  = jax.random.normal(key=jax.random.PRNGKey(0), shape=(4, 334, seq_len, seq_len))
causal_mask = jnp.triu(
            jnp.ones((seq_len, seq_len)) * -jnp.inf, k=1)
        # Broadcast causal_mask to match attn_weights shape: [1, 1, seq_len, seq_len]
att_w = att_w + causal_mask[None, None, :, :]
att_w

Array([[[[ 1.62264216e+00,            -inf,            -inf, ...,
                     -inf,            -inf,            -inf],
         [ 6.59807801e-01,  1.06801593e+00,            -inf, ...,
                     -inf,            -inf,            -inf],
         [ 6.90909326e-01, -2.56064266e-01, -2.02274013e+00, ...,
                     -inf,            -inf,            -inf],
         ...,
         [ 1.60733953e-01, -1.94400907e-01, -3.44724953e-01, ...,
          -9.06192541e-01,            -inf,            -inf],
         [-5.38808182e-02,  4.39482003e-01,  4.41510558e-01, ...,
          -4.53171551e-01,  6.92858279e-01,            -inf],
         [-1.02863717e+00,  1.94518065e+00,  1.84248701e-01, ...,
          -3.22564431e-02, -2.18967843e+00, -6.30790830e-01]],

        [[-3.63802999e-01,            -inf,            -inf, ...,
                     -inf,            -inf,            -inf],
         [-1.63032621e-01,  9.34063017e-01,            -inf, ...,
                     -

In [57]:
att_m = attention_mask[:4]
att_m = att_m.astype(bool)
att_m

Array([[False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  Tru

In [64]:
from flax import linen as nn

att_wp = nn.softmax(att_w, axis=-1)
att_wp = jnp.where(att_m[:, None, None, :], att_wp, 0.0)
att_wp

Array([[[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.00354702,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.00546193,
          0.01718147, 0.        ],
         [0.        , 0.        , 0.        , ..., 0.00707725,
          0.00081829, 0.00388978]],

        [[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.01599434,
          0.        , 0. 

In [69]:
att_w.diagonal(axis1=-2, axis2=-1).at[attention_mask[:, None, :]].set(0)

2025-09-25 18:08:20.511055: E external/xla/xla/service/gpu/gpu_hlo_schedule.cc:654] The byte size of input/output arguments (15901740004) exceeds the base limit (12541722624). This indicates an error in the calculation!
2025-09-25 18:08:20.511189: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3021] Can't reduce memory use below 0B (0 bytes) by rematerialization; only reduced to 14.81GiB (15901740000 bytes), down from 14.81GiB (15901740000 bytes) originally
2025-09-25 18:08:30.541093: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 14.81GiB (rounded to 15901740032)requested by op 
2025-09-25 18:08:30.544588: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ********_______***______________******___***********************____________________________________
E0925 18:08:30.544630  100656 pjrt_stream_executor_client.cc:3077] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while 

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 15901740000 bytes.

In [71]:
from src.subspace_curve import LoRACategorySubspace
s_model = LoRACategorySubspace(lora_mask=lora_mask, r=8,
                               model=model, k=0, weight_decay=0., 
                               perm_spec=None,
                               natural_parameterization=False,
                               mutuable_param_name=False)
key = jax.random.PRNGKey(0)
curve_mask = jax.tree.map(lambda x: False, params['params'])
s_params = s_model.init_params_from_point(key, params, curve_mask, jitter=1e-2)
s_params['params']['layers_0']['self_attn']['q_proj']['kernel']

NameError: name 'lora_mask' is not defined

In [54]:
s_params['params']['layers_0']['self_attn']['q_proj']['kernel'].A.shape

(8, 896)

In [70]:
train_mask = s_model.update_mask_for_lora(train_mask, s_params['params'], b_off=False, w0_off=True)
s_model.set_train_mask(train_mask)
s_model.train_mask

{'embed_tokens': {'embedding': False},
 'layers_0': {'input_layernorm': {'weight': False},
  'mlp': {'down_proj': {'kernel': False},
   'gate_proj': {'kernel': False},
   'up_proj': {'kernel': False}},
  'post_attention_layernorm': {'weight': False},
  'self_attn': {'k_proj': {'bias': False, 'kernel': False},
   'o_proj': {'kernel': False},
   'q_proj': {'bias': True,
    'kernel': LoRAParams(A=True, B=True, w0=False) with dims=(0, 1))},
   'v_proj': {'bias': False,
    'kernel': LoRAParams(A=True, B=True, w0=False) with dims=(0, 1))}}},
 'layers_1': {'input_layernorm': {'weight': False},
  'mlp': {'down_proj': {'kernel': False},
   'gate_proj': {'kernel': False},
   'up_proj': {'kernel': False}},
  'post_attention_layernorm': {'weight': False},
  'self_attn': {'k_proj': {'bias': False, 'kernel': False},
   'o_proj': {'kernel': False},
   'q_proj': {'bias': True,
    'kernel': LoRAParams(A=True, B=True, w0=False) with dims=(0, 1))},
   'v_proj': {'bias': False,
    'kernel': LoRAParams

In [71]:
only_train_params = s_model.only_trainable_params(s_params['params'])
n_params = sum(p.size for p in jax.tree.flatten(only_train_params)[0])
print(f"Number of trainable parameters: {n_params:_}")

Number of trainable parameters: 562_176


In [76]:
params['params']['layers_0']['self_attn']['v_proj']['kernel'].shape

(896, 128)

In [74]:
print(s_params['params']['layers_0']['self_attn']['q_proj']['kernel'].A.shape)
print(s_params['params']['layers_0']['self_attn']['v_proj']['kernel'].A.shape)

(8, 896)
(8, 128)


In [78]:
(896*8*2+(896*8+128*8))*24+896*24

562176

In [51]:
562176/24/8

2928.0

In [ ]:
896*8*3*24

516096

In [65]:
896*8*24*2+896*24

365568

In [31]:
only_train_params

{'embed_tokens': {'embedding': Array([], shape=(0,), dtype=float32)},
 'layers_0': {'input_layernorm': {'weight': Array([], shape=(0,), dtype=float32)},
  'mlp': {'down_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'gate_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'up_proj': {'kernel': Array([], shape=(0,), dtype=float32)}},
  'post_attention_layernorm': {'weight': Array([], shape=(0,), dtype=float32)},
  'self_attn': {'k_proj': {'bias': Array([], shape=(0,), dtype=float32),
    'kernel': Array([], shape=(0,), dtype=float32)},
   'o_proj': {'kernel': Array([], shape=(0,), dtype=float32)},
   'q_proj': {'bias': Array([], shape=(0,), dtype=float32),
    'kernel': Array([[ 0.002533,  0.01355 , -0.01306 , ..., -0.10986 , -0.01636 ,
             0.01843 ],
           [-0.000248,  0.00885 , -0.02087 , ...,  0.08105 ,  0.0635  ,
             0.05737 ],
           [ 0.0155  ,  0.03345 ,  0.0459  , ...,  0.01453 ,  0.0586  ,
             0.05786 ],
           ...,


In [6]:
nll, state, out = s_model.nll(params=s_params['params'], state={}, t=0., x=(input_ids[:4], attention_mask[:4]), y=labels[:4], train=True)

In [88]:
nll, state, out = s_model.nll(params=s_params['params'], state={}, t=0., x=input_ids[:4], y=labels[:4], train=True)

In [19]:
max_seqlen = jnp.argmax(attention_mask[:4], axis=1).min()
input_ids = input_ids[:4, max_seqlen:]

In [23]:
input_ids

Array([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643,   4418,    279,  21085,
          3685,    323,   4226,    279,   3405,    448,    279,   4244,
           364,   1866,      6,    476,    364,   3849,  29636,  12187,
           424,     25,  49861,  46225, 144936,     79, 148951, 145231,
            81, 146159,  98538,     77,     11,    481, 145388,  98538,
            77,     14,    701,   1083,   3881,    553,   1181,    835,
          7831,    434,  60014,    320, 127521,  20064,  14391,    282,
         30657,   5428,     72,    320,     69, 146118, 145231, 144788,
        144936,   6321, 145231,      8,    320,   8844,   5731,    374,
           825,    315,    279,  10867,  27401,  15459,   2878,    279,
         75885,  21477,   6576,   1103,   8870,    315,    279,  75885,
         87635,   4128,   2997,     13,   1084,    374,  15503,  21355,
           304,  10259,     11,  20500,    320,  32812,    398, 

In [12]:
input_ids.shape

(9400, 525)

In [9]:
out

Array([[ 8.62,  9.96],
       [10.13, 11.75],
       [ 9.06, 10.86],
       [ 9.12, 10.59]], dtype=float16)

In [10]:
labels[:4]

Array([1, 1, 1, 1], dtype=int32)

In [11]:
jnp.argmax(out, axis=-1) == labels[:4]

Array([ True,  True,  True,  True], dtype=bool)

In [ ]:
state

{}

In [91]:
nll

Array(0.193, dtype=float16)